In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
try:
    import lm_eval
except ImportError:
    %pip install -q git+https://github.com/EleutherAI/lm-evaluation-harness
    import lm_eval

# Constants

In [ ]:
TAWJEEH_DATASET_NAME = 'Arabic_Dialects_Dataset'
HF_EXPERIMENTAL_DATASET_NAME = 'MagedSaeed/arabic_dialects_dataset_experimental'
TASK_NAME='dialect_identification'
MODEL_PATH = "/hdd/shared_models/jais-13b"
BATCH_SIZE = 4
# ------------------------
HF_EXPERIMENTAL_DATASET_NAME_USED_FOR_TUNING = 'MagedSaeed/arabench_dev_experimental'
TAWJEEH_DATASET_NAME_USED_FOR_TUNING = 'AraBench_dev'
# -----------------------
TUNED_MODEL_PATH = None

In [4]:
MODEL_NAME = MODEL_PATH.split('/')[-1]
TOKENIZER_PATH = MODEL_PATH

# Building the prompts dataset

In [5]:
import requests
 
from tqdm.auto import tqdm
 
prompts = None
 
tries = 10
for i in tqdm(range(tries)):
    api_response = requests.get(url='https://tawjeeh.up.railway.app/api/prompt/list?project_secret_key=6Wirj')
    if api_response.ok:
        prompts = api_response.json()
        break
if not prompts:
    raise Exception('Failed to fetch prompts')
prompts

  0%|          | 0/10 [00:00<?, ?it/s]

[{'id': 14838,
  'tags': [],
  'name': 'Sarcasm Detection-sarcasmCues',
  'task': {'name': 'sarcasm detection'},
  'status': 'SUBMITTED',
  'template': 'Task: Identify whether the following tweet is sarcastic or non-sarcastic by comparing it to typical sarcasm cues.\r\n\r\nIndicators of Sarcasm:\r\n- Exaggerated praise or criticism\r\n- Statements that imply the opposite of their literal meaning\r\n- Use of irony or unexpected humor\r\n\r\nTweet: {{ tweet }}\r\n\r\nAnswer: Based on the indicators, respond with "sarcastic" if the tweet shows sarcasm, or "non-sarcastic" if it does not.\r\n|||\r\n{{ answer_choices[label] }}',
  'dataset_name': 'arbml/ArSarcasm_v2',
  'dataset_subset': 'default',
  'answer_choices': ['Non sarcastic', 'sarcastic'],
  'text_direction': 'ltr'},
 {'id': 14837,
  'tags': [],
  'name': 'Sarcasm Detection-COT',
  'task': {'name': 'sarcasm detection'},
  'status': 'SUBMITTED',
  'template': 'Task: Determine if the following tweet contains sarcasm by following thes

filter prompts:
- get only the approved ones
- get only the ones on the sarcasim detection datasets (emotone_ar,sem_eval_2018_task_1)

In [6]:
filtered_prompts = list(filter(lambda prompt: prompt['status'] == 'APPROVED', prompts))
len(filtered_prompts)

153

### Get the dataset prompts

In [7]:
# you can either filter by task or dataset
dataset_prompts = list(
    filter(
        lambda prompt: TAWJEEH_DATASET_NAME in prompt['dataset_name'],
        filtered_prompts,
    )
)
len(dataset_prompts)

4

### Download the experimental dataset

In [8]:
import datasets

In [9]:
hf_exp_dataset = datasets.load_dataset(HF_EXPERIMENTAL_DATASET_NAME)
hf_exp_dataset

DatasetDict({
    test: Dataset({
        features: ['Text', 'label'],
        num_rows: 9992
    })
})

In [10]:
dataset_prompts

[{'id': 14784,
  'tags': [],
  'name': 'Dialect based on cultural references',
  'task': {'name': 'dialect identification'},
  'status': 'APPROVED',
  'template': 'For the following Arabic text: {{Text}}, the most probable dialect based on cultural references that may indicate a specific region (among Levant, North Africa, Egypt, GULF, MSA) is:\r\n|||\r\n{{answer_choices[label]}}',
  'dataset_name': 'arbml/Arabic_Dialects_Dataset',
  'dataset_subset': 'default',
  'answer_choices': ['Levant', 'North Africa', 'Egypt', 'GULF', 'MSA'],
  'text_direction': 'ltr'},
 {'id': 14783,
  'tags': [],
  'name': 'Dialect based on unique words or phrases',
  'task': {'name': 'dialect identification'},
  'status': 'APPROVED',
  'template': 'For the following Arabic text: {{Text}}, the most probable dialect (among Levant, North Africa, Egypt, GULF, MSA) based on unique words or phrases specific to a dialect is:\r\n|||\r\n{{answer_choices[label]}}',
  'dataset_name': 'arbml/Arabic_Dialects_Dataset',
  '

### Merge the prompts

In [11]:
from jinja2 import Environment, StrictUndefined

In [12]:
def apply_template(prompt_template, sample):
    template = prompt_template['template']
    sample['answer_choices'] = prompt_template['answer_choices']
    env = Environment(undefined=StrictUndefined)
    if "|||" not in template:
        raise ValueError("No ||| dividor")
    template = env.from_string(template)
    rendered_template = template.render(**sample)
    return rendered_template

Perform generation on one example prompt, for experimentation

In [13]:
example_prompt_template = dataset_prompts[0]
print(apply_template(example_prompt_template, hf_exp_dataset['test'][2]))

For the following Arabic text: أنا لو عيز أصلح في بلدي هما ملهم ميشطروا على اللى عندهم أل أيه عايزنا نحترم حرية التعبير .طيب أنتو كمان أحترموا حرية العقيده . و خلوا المسلمين اللى عندكوا يشاركوا في الأنتخابات, the most probable dialect based on cultural references that may indicate a specific region (among Levant, North Africa, Egypt, GULF, MSA) is:
|||
Egypt


In [14]:
example_prompt_template['answer_choices']

['Levant', 'North Africa', 'Egypt', 'GULF', 'MSA']

In [15]:
for prompt in dataset_prompts:
    prompt['merged_samples'] = list(
        map(
            lambda sample: apply_template(prompt, sample),
            # hf_exp_dataset['test'].select(range(100)),
            tqdm(hf_exp_dataset['test']),
        )
    )
len(dataset_prompts)

  0%|          | 0/9992 [00:00<?, ?it/s]

  0%|          | 0/9992 [00:00<?, ?it/s]

  0%|          | 0/9992 [00:00<?, ?it/s]

  0%|          | 0/9992 [00:00<?, ?it/s]

4

# Load LLM and Prepare

In [16]:
from datasets import DatasetDict

def create_hf_dataset(examples, columns = ['text', 'label']):
  texts = []
  labels = []
  for example in examples:
    prefix= example.split('|||')[0].replace('\n', '')
    output = example.split('|||')[1].replace('\n', '')
    texts.append(prefix)
    labels.append(output)
  dataset = DatasetDict({ 'test' : datasets.Dataset.from_dict({
      columns[0]: texts,
      columns[1]: labels,
  })})
  return dataset

In [17]:
from lm_eval.models.huggingface import HFLM
from transformers import AutoModelForCausalLM, AutoTokenizer

In [18]:
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, trust_remote_code=True, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)

2024-11-07:13:21:01,943 INFO     [modeling.py:1014] We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

In [19]:
from peft import PeftModel
if TUNED_MODEL_PATH:
    print('loading tuned model')
    model = PeftModel.from_pretrained(model,TUNED_MODEL_PATH)

In [20]:
lm_obj = HFLM(
    pretrained=model,
    trust_remote_code=True,
    # parallelize=True,
    device_map="auto",
    tokenizer=tokenizer,
    batch_size=BATCH_SIZE,
)

2024-11-07:13:22:46,712 WARNING  [huggingface.py:96] `pretrained` model kwarg is not of type `str`. Many other model arguments may be ignored. Please do not launch via accelerate or use `parallelize=True` if passing an existing model this way.
2024-11-07:13:22:46,714 INFO     [huggingface.py:494] Model type cannot be determined. Using default model type 'default'
2024-11-07:13:22:46,725 WARNING  [huggingface.py:277] Passed an already-initialized model through `pretrained`, assuming single-process call to evaluate() or custom distributed integration


In [21]:
def evaluate_tasks(tasks,dataset_sub_path=TAWJEEH_DATASET_NAME):
    # MAKE SURE THE NOTEBOOK IS RUNNING FROM THE PROJECT ROOT!
    task_manager = lm_eval.tasks.TaskManager(include_path=f"eval_harness_extra_tasks/{dataset_sub_path}")
    results = lm_eval.simple_evaluate(  # call simple_evaluate
        model=lm_obj,
        tasks=tasks,
        num_fewshot=0,
        task_manager=task_manager,
    )
    return results

In [22]:
import json

def create_and_evaluate_single_prompt(prompt):
    prompt_id = prompt['id']
    if TUNED_MODEL_PATH:
        results_dir = f'evaluation_results/{MODEL_NAME}_tuned/{TASK_NAME}/{TAWJEEH_DATASET_NAME}'
    else:
      results_dir = f'evaluation_results/{MODEL_NAME}/{TASK_NAME}/{TAWJEEH_DATASET_NAME}'
    prompt_results_file_path = f'{results_dir}/prompt_{prompt_id}.json'
    
    # Skip if results already exist
    if os.path.exists(prompt_results_file_path) and os.path.getsize(prompt_results_file_path) > 0:
        print(f"Skipping prompt {prompt_id} - results already exist")
        with open(prompt_results_file_path, 'r') as f:
            return json.load(f)
    
    # Create dataset and task files
    merged_samples = prompt['merged_samples']
    answer_choices = prompt['answer_choices']
    dataset = create_hf_dataset(merged_samples)
    
    # Save dataset
    dataset_dir = f'experimental_hf_datasets/{TAWJEEH_DATASET_NAME}/prompt_{prompt_id}'
    os.makedirs(dataset_dir, exist_ok=True)
    dataset['test'].to_parquet(f"{dataset_dir}/data.parquet")
    
    # Create YAML configuration
    yaml_text = f'''task: {TAWJEEH_DATASET_NAME}_prompt_{prompt_id}
dataset_path: experimental_hf_datasets/{TAWJEEH_DATASET_NAME}/prompt_{prompt_id}
output_type: multiple_choice
test_split: train
doc_to_text: text
doc_to_target: label
doc_to_choice: {answer_choices}
metric_list:
  - metric: acc
    aggregation: mean
    higher_is_better: True
  - metric: acc_norm
    aggregation: mean
    higher_is_better: true
metadata:
  version: 1.0'''
    
    # Save YAML
    yaml_dir = f'eval_harness_extra_tasks/{TAWJEEH_DATASET_NAME}'
    os.makedirs(yaml_dir, exist_ok=True)
    with open(f'{yaml_dir}/prompt_{prompt_id}.yaml', 'w') as f:
        f.write(yaml_text)
    
    # Evaluate single prompt
    evaluation_task_name = f'{TAWJEEH_DATASET_NAME}_prompt_{prompt_id}'
    prompt_results = evaluate_tasks(tasks=[evaluation_task_name])
    
    print(lm_eval.utils.make_table(prompt_results))
    
    # Save results immediately
    os.makedirs(results_dir, exist_ok=True)
    with open(prompt_results_file_path, 'w') as f:
        json.dump(prompt_results, f, ensure_ascii=False, indent=4, 
                 default=lambda o: '<not serializable>')
    
    print(f"Completed evaluation for prompt {prompt_id}")
    return prompt_results

In [23]:
def evaluate_all_prompts_sequentially(prompts):
    print(f"Starting sequential evaluation of {len(prompts)} prompts")
    all_results = {}
    
    for i, prompt in enumerate(prompts, 1):
        print('-' * 80)
        print(f"\nProcessing prompt {i}/{len(prompts)}")
        print("Template:", prompt['template'])
        print('-' * 80)
        
        prompt_results = create_and_evaluate_single_prompt(prompt)
        all_results[f"{TAWJEEH_DATASET_NAME}_prompt_{prompt['id']}"] = prompt_results
    
    return {'results': all_results}

# Evaluate

In [ ]:
evaluate_all_prompts_sequentially(prompts=dataset_prompts)

Starting sequential evaluation of 4 prompts
--------------------------------------------------------------------------------

Processing prompt 1/4
Template: For the following Arabic text: {{Text}}, the most probable dialect based on cultural references that may indicate a specific region (among Levant, North Africa, Egypt, GULF, MSA) is:
|||
{{answer_choices[label]}}
--------------------------------------------------------------------------------


Creating parquet from Arrow format:   0%|          | 0/10 [00:00<?, ?ba/s]

2024-11-07:13:22:53,868 INFO     [evaluator.py:164] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2024-11-07:13:22:53,870 INFO     [evaluator.py:217] Using pre-initialized model


Generating train split: 0 examples [00:00, ? examples/s]

2024-11-07:13:22:53,940 WARNING  [task.py:325] [Task: Arabic_Dialects_Dataset_prompt_14784] has_training_docs and has_validation_docs are False, using test_docs as fewshot_docs but this is not recommended.
2024-11-07:13:22:53,941 WARNING  [task.py:325] [Task: Arabic_Dialects_Dataset_prompt_14784] has_training_docs and has_validation_docs are False, using test_docs as fewshot_docs but this is not recommended.
2024-11-07:13:22:54,106 WARNING  [evaluator.py:270] Overwriting default num_fewshot of Arabic_Dialects_Dataset_prompt_14784 from None to 0
2024-11-07:13:22:54,107 INFO     [task.py:415] Building contexts for Arabic_Dialects_Dataset_prompt_14784 on rank 0...
100%|██████████| 9992/9992 [00:00<00:00, 36100.27it/s]
2024-11-07:13:22:54,553 INFO     [evaluator.py:489] Running loglikelihood requests
Token indices sequence length is longer than the specified maximum sequence length for this model (3711 > 2048). Running this sequence through the model will result in indexing errors
Running 

# Evaluation on the prompts of the dataset used for tuning

## Building the prompts

In [ ]:
dataset_used_for_tuning_prompts = list(
    filter(
        lambda prompt: TAWJEEH_DATASET_NAME_USED_FOR_TUNING in prompt['dataset_name'],
        filtered_prompts,
    )
)
len(dataset_used_for_tuning_prompts)

In [ ]:
hf_exp_dataset_used_for_tuning = datasets.load_dataset(HF_EXPERIMENTAL_DATASET_NAME_USED_FOR_TUNING)
hf_exp_dataset_used_for_tuning

In [ ]:
example_prompt_template_from_tuning_dataset = dataset_used_for_tuning_prompts[0]
print(apply_template(example_prompt_template_from_tuning_dataset, hf_exp_dataset_used_for_tuning['test'][2]))

In [ ]:
for prompt in dataset_used_for_tuning_prompts:
    prompt['merged_samples'] = list(
        map(
            lambda sample: apply_template(prompt, sample),
            # hf_exp_dataset['test'].select(range(100)),
            hf_exp_dataset['test'],
        )
    )

## Evaluate

In [ ]:
evaluate_all_prompts_sequentially(prompts=dataset_used_for_tuning_prompts)